# TI-DPO-compatible end-to-end evaluation suite

This Kaggle notebook runs local Hugging Face causal
language models over MMLU, GSM8K, full GPQA Main, HumanEval, TruthfulQA MC2, and IFEval,
saves durable raw generations or every candidate log-likelihood, and computes the complete
benchmark suite locally from those frozen artifacts.

It never calls an LLM judge or external inference API. HumanEval `pass@1` is calculated by
executing generated Python against the official tests in timed worker processes; generated
code is untrusted, so the notebook creates a separate raw-artifact ZIP before that step.
Attach model directories as Kaggle Inputs, enable a GPU accelerator (T4 x2 recommended),
edit only the configuration cell below, and run all cells.


## 1. Configuration

This is the only cell intended for routine editing. Add or remove model dictionaries.
A per-model `apply_chat_template` boolean may override the global setting. Optional advanced
fields are `chat_template` (the name of an official tokenizer template when several exist),
`trust_remote_code`, `batch_size`, and `max_batch_size`.


In [1]:
MODELS = [
    {
        "name": "VPDPO_B_Norm_DPO",
        "path": "/kaggle/input/datasets/lonnyng/b-norm-dpo-olmo-1b/olmo2_bees_b_norm_dpo/run/final",
    },
    {
        "name": "VPDPO_B_Norm_VDPO",
        "path": "/kaggle/input/datasets/ahmadsubhaniiqbal/b-norm-vpdpo-olmo-1b/olmo2_bees_b_norm_vdpo/run/final",
    },
    {
        "name": "VPDPO_B_DPO",
        "path": "/kaggle/input/datasets/ahmadsubhaniiqbal/olmo-1b-b-d",
    },
    {
        "name": "VPDPO_B_VDPO",
        "path": "/kaggle/input/datasets/ahmadsubhaniiqbal/olmo-1b-b-vdpo",
    },
    {
        "name": "VPDPO_C_DPO",
        "path": "/kaggle/input/datasets/ahmadsubhaniiqbal/olmo-1b-c-dpo",
    },
    {
        "name": "VPDPO_C_VDPO",
        "path": "/kaggle/input/datasets/ahmadsubhaniiqbal/olmo-1b-c-vdpo",
    },
    {
        "name": "Simple_DPO",
        "path": "/kaggle/input/datasets/ahmadsubhaniiqbal/olmo-1b-dpo-plain",
    },
    {
        "name": "VPDPO_A",
        "path": "/kaggle/input/datasets/ahmadsubhaniiqbal/olmo-1b-method-a",
    },
    {
        "name": "SimPO",
        "path": "/kaggle/input/datasets/jonathonbreaux/olmo2-bees-simpo/olmo2_bees_simpo/run/final",
    },
    {
        "name": "SAMPO",
        "path": "/kaggle/input/datasets/lonnyng/sampo-models/olmo2_bees_sampo/run/final",
    },
    {
        "name": "TIDPO",
        "path": "/kaggle/input/datasets/ahmadsubhaniiqbal/olmo-bees-tidpo/olmo2_bees_tidpo/run/final",
    },
]

OUTPUT_ROOT = "/kaggle/working/tidpo_generations"
HF_TOKEN_SECRET_NAME = "Huggingface"
SUITE_NAME = "tidpo_compatible_v1"

APPLY_CHAT_TEMPLATE = True
SEED = 42

# End-to-end local scoring. HumanEval requires executing untrusted generated Python.
RUN_SCORING = True
EXECUTE_HUMANEVAL = True
HUMANEVAL_NUM_WORKERS = 4
HUMANEVAL_TIMEOUT_SECONDS = 3.0

# Conservative defaults for T4 memory. The harness auto-tunes request batches.
DEFAULT_BATCH_SIZE = "auto:4"
DEFAULT_MAX_BATCH_SIZE = 16
SAMPLE_CHUNK_SIZE = 16
FLUSH_EVERY = 8


## 2. Dependency installation

The evaluator logic is pinned to lm-evaluation-harness v0.4.9.2 at an exact Git commit.
Core userspace dependencies are pinned as well. Kaggle's CUDA-enabled PyTorch build is
intentionally retained instead of being replaced; its exact version is recorded later.
Internet access must be enabled for this installation and the benchmark downloads.


In [2]:
import subprocess
import sys

LM_EVAL_COMMIT = "ad3f4d0cad1cfcdb815f1e795f7947e49ed9f2e9"
LM_EVAL_VERSION = "0.4.9.2"
CODE_EVAL_REVISION = "262b7e74cf29a715d74f8b02ba1d6ef74e432333"

PINNED_PACKAGES = [
    "transformers==4.57.1",
    "accelerate==1.11.0",
    "datasets==4.4.1",
    "evaluate==0.4.6",
    "huggingface-hub==0.36.0",
    "tokenizers==0.22.1",
    "safetensors==0.6.2",
    "peft==0.17.1",
    "sentencepiece==0.2.1",
    "jsonlines==4.0.0",
    "langdetect==1.0.9",
    "immutabledict==4.2.1",
    "nltk==3.9.1",
    "rouge-score==0.1.2",
    "sacrebleu==2.5.1",
    "sqlitedict==2.1.0",
    "word2number==1.1",
    "more-itertools==10.6.0",
    "zstandard==0.23.0",
    "dill==0.4.0",
    "pytablewriter==1.2.1",
    "tqdm-multiprocess==0.0.11",
    "numexpr==2.10.2",
    "pybind11==2.13.6",
    "scikit-learn==1.6.1",
]

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", "--upgrade"]
    + PINNED_PACKAGES
)
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--disable-pip-version-check",
        "--no-deps",
        "--force-reinstall",
        f"git+https://github.com/EleutherAI/lm-evaluation-harness.git@{LM_EVAL_COMMIT}",
    ]
)
print("Pinned end-to-end evaluation environment installed:")
print("\n".join(f"  - {package}" for package in PINNED_PACKAGES))
print(f"  - lm-evaluation-harness @ {LM_EVAL_COMMIT}")
print(f"  - evaluate-metric/code_eval @ {CODE_EVAL_REVISION}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 31.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.8/485.8 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
blosc2 4.1.2 requires numexpr>=2.14.1; platform_machine != "wasm32", but you have numexpr 2.10.2 which is incompatible.


Pinned end-to-end evaluation environment installed:
  - transformers==4.57.1
  - accelerate==1.11.0
  - datasets==4.4.1
  - evaluate==0.4.6
  - huggingface-hub==0.36.0
  - tokenizers==0.22.1
  - safetensors==0.6.2
  - peft==0.17.1
  - sentencepiece==0.2.1
  - jsonlines==4.0.0
  - langdetect==1.0.9
  - immutabledict==4.2.1
  - nltk==3.9.1
  - rouge-score==0.1.2
  - sacrebleu==2.5.1
  - sqlitedict==2.1.0
  - word2number==1.1
  - more-itertools==10.6.0
  - zstandard==0.23.0
  - dill==0.4.0
  - pytablewriter==1.2.1
  - tqdm-multiprocess==0.0.11
  - numexpr==2.10.2
  - pybind11==2.13.6
  - scikit-learn==1.6.1
  - lm-evaluation-harness @ ad3f4d0cad1cfcdb815f1e795f7947e49ed9f2e9
  - evaluate-metric/code_eval @ 262b7e74cf29a715d74f8b02ba1d6ef74e432333


## 3. Environment diagnostics

This cell imports the installed stack, freezes random seeds, creates workspace-local caches,
and reports every visible GPU. T4 GPUs use FP16; BF16 is selected only when all visible GPUs
safely support it.


In [3]:
import csv
import gc
import hashlib
import importlib.metadata
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import traceback
import warnings
import zipfile
from collections import defaultdict
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path

OUTPUT_ROOT_PATH = Path(OUTPUT_ROOT).expanduser().resolve()
OUTPUT_ROOT_PATH.mkdir(parents=True, exist_ok=True)
CACHE_ROOT = (
    Path("/kaggle/working/.cache/tidpo_generation")
    / SUITE_NAME
    / f"lm-eval-{LM_EVAL_COMMIT[:12]}"
    / f"seed-{SEED}"
)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(CACHE_ROOT / "huggingface")
os.environ["HF_DATASETS_CACHE"] = str(CACHE_ROOT / "huggingface/datasets")
os.environ["TRANSFORMERS_CACHE"] = str(CACHE_ROOT / "huggingface/transformers")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import accelerate
import datasets
import evaluate
import huggingface_hub
import lm_eval
import numpy as np
import torch
import transformers
from tqdm.auto import tqdm


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


seed_everything(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. In Kaggle, select Settings > Accelerator > GPU T4 x2.")

GPU_INFO = []
for gpu_index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(gpu_index)
    item = {
        "index": gpu_index,
        "name": props.name,
        "total_memory_bytes": int(props.total_memory),
        "compute_capability": [int(props.major), int(props.minor)],
    }
    GPU_INFO.append(item)
    print(
        f"GPU {gpu_index}: {props.name} | {props.total_memory / 2**30:.2f} GiB | "
        f"compute capability {props.major}.{props.minor}"
    )

if torch.cuda.device_count() < 2:
    warnings.warn("Only one GPU is visible. The notebook supports it, but Kaggle T4 x2 is recommended.")
elif all("T4" in item["name"] for item in GPU_INFO[:2]):
    print("Kaggle dual-T4 environment detected.")

all_bf16 = bool(
    torch.cuda.is_bf16_supported()
    and all(item["compute_capability"][0] >= 8 for item in GPU_INFO)
)
MODEL_DTYPE = torch.bfloat16 if all_bf16 else torch.float16
MODEL_DTYPE_NAME = "bfloat16" if all_bf16 else "float16"

PACKAGE_VERSIONS = {
    "python": sys.version,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "evaluate": evaluate.__version__,
    "accelerate": accelerate.__version__,
    "huggingface_hub": huggingface_hub.__version__,
    "lm_eval": importlib.metadata.version("lm_eval"),
    "lm_eval_commit": LM_EVAL_COMMIT,
    "lm_eval_module": str(Path(lm_eval.__file__).resolve()),
}
lm_eval_distribution = importlib.metadata.distribution("lm_eval")
direct_url_text = lm_eval_distribution.read_text("direct_url.json")
LM_EVAL_DIRECT_URL = json.loads(direct_url_text) if direct_url_text else {}
installed_lm_eval_commit = LM_EVAL_DIRECT_URL.get("vcs_info", {}).get("commit_id")
if PACKAGE_VERSIONS["lm_eval"] != LM_EVAL_VERSION or installed_lm_eval_commit != LM_EVAL_COMMIT:
    raise RuntimeError(
        "Pinned lm-evaluation-harness verification failed: "
        f"version={PACKAGE_VERSIONS['lm_eval']!r}, commit={installed_lm_eval_commit!r}"
    )
print(json.dumps(PACKAGE_VERSIONS, indent=2))
print("Inference dtype:", MODEL_DTYPE_NAME)


/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


GPU 0: Tesla T4 | 14.56 GiB | compute capability 7.5
GPU 1: Tesla T4 | 14.56 GiB | compute capability 7.5
Kaggle dual-T4 environment detected.
{
  "python": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]",
  "torch": "2.10.0+cu128",
  "cuda": "12.8",
  "transformers": "4.57.1",
  "datasets": "4.4.1",
  "evaluate": "0.4.6",
  "accelerate": "1.11.0",
  "huggingface_hub": "0.36.0",
  "lm_eval": "0.4.9.2",
  "lm_eval_commit": "ad3f4d0cad1cfcdb815f1e795f7947e49ed9f2e9",
  "lm_eval_module": "/usr/local/lib/python3.12/dist-packages/lm_eval/__init__.py"
}
Inference dtype: float16


## 4. Authentication

GPQA is gated. Accept the dataset's terms on Hugging Face, add a Kaggle secret named
`HF_TOKEN`, and grant the notebook access to it. Missing authentication is reported without
ever printing the token.


In [4]:
HF_TOKEN_AVAILABLE = False
try:
    from kaggle_secrets import UserSecretsClient

    hf_token = UserSecretsClient().get_secret(HF_TOKEN_SECRET_NAME)
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        HF_TOKEN_AVAILABLE = True
        print(f"Hugging Face authentication loaded from Kaggle Secret {HF_TOKEN_SECRET_NAME!r}.")
    del hf_token
except Exception as exc:
    warnings.warn(
        f"Hugging Face token was not loaded ({type(exc).__name__}). "
        "Public tasks can run, but GPQA Main may fail until its terms are accepted and the secret is attached."
    )


Hugging Face authentication loaded from Kaggle Secret 'Huggingface'.


## 5. Frozen benchmark configuration

The harness and every source dataset are pinned to immutable commits. `gpqa_main_zeroshot`
is explicitly guarded against Diamond. Generation limits are deliberate safety limits;
deterministic decoding uses one sequence with sampling disabled. The runtime protocol hash
also incorporates resolved task configs, source fingerprints, and expected sample IDs.


In [5]:
DATASET_REVISIONS = {
    "mmlu": "c30699e8356da336a370243923dbaf21066bb9fe",
    "gsm8k": "740312add88f781978c0658806c59bc2815b9866",
    "gpqa": "633f5ee89ab8ad4522a9f850766b73f62147ffdd",
    "humaneval": "7dce6050a7d6d172f3cc5c32aa97f52fa1a2e544",
    "truthfulqa": "741b8276f2d1982aa3d5b832d3ee81ed3b896490",
    "ifeval": "966cd89545d6b6acfd7638bc708b98261ca58e84",
}

BENCHMARK_PROTOCOLS = {
    "mmlu": {
        "harness_task": "mmlu",
        "dataset_path": "cais/mmlu",
        "dataset_revision": DATASET_REVISIONS["mmlu"],
        "variant": "original full 57-subject MMLU",
        "output_type": "multiple_choice",
        "num_fewshot": 5,
        "generation_kwargs": None,
    },
    "gsm8k": {
        "harness_task": "gsm8k",
        "dataset_path": "openai/gsm8k",
        "dataset_revision": DATASET_REVISIONS["gsm8k"],
        "variant": "main/test",
        "output_type": "generate_until",
        "num_fewshot": 5,
        "generation_kwargs": {
            "until": ["Question:", "</s>", "<|im_end|>"],
            "do_sample": False,
            "temperature": 0.0,
            "max_gen_toks": 512,
        },
    },
    "gpqa": {
        "harness_task": "gpqa_main_zeroshot",
        "dataset_path": "Idavidrein/gpqa",
        "dataset_revision": DATASET_REVISIONS["gpqa"],
        "dataset_name": "gpqa_main",
        "variant": "FULL GPQA Main; never Diamond",
        "output_type": "multiple_choice",
        "num_fewshot": 0,
        "generation_kwargs": None,
    },
    "humaneval": {
        "harness_task": "humaneval_generation_only",
        "canonical_harness_task": "humaneval",
        "dataset_path": "openai/openai_humaneval",
        "dataset_revision": DATASET_REVISIONS["humaneval"],
        "variant": "OpenAI HumanEval all 164 problems; generation only",
        "output_type": "generate_until",
        "num_fewshot": 0,
        "generation_kwargs": {
            "until": ["\nclass", "\ndef", "\n#", "\nif", "\nprint"],
            "do_sample": False,
            "temperature": 0.0,
            "max_gen_toks": 1024,
        },
    },
    "truthfulqa": {
        "harness_task": "truthfulqa_mc2",
        "dataset_path": "truthful_qa",
        "dataset_revision": DATASET_REVISIONS["truthfulqa"],
        "dataset_name": "multiple_choice",
        "variant": "original TruthfulQA MC2",
        "output_type": "multiple_choice",
        "num_fewshot": 0,
        "generation_kwargs": None,
    },
    "ifeval": {
        "harness_task": "ifeval",
        "dataset_path": "google/IFEval",
        "dataset_revision": DATASET_REVISIONS["ifeval"],
        "variant": "original Google IFEval",
        "output_type": "generate_until",
        "num_fewshot": 0,
        "generation_kwargs": {
            "until": [],
            "do_sample": False,
            "temperature": 0.0,
            "max_gen_toks": 1280,
        },
    },
}
BENCHMARK_ORDER = ["mmlu", "gsm8k", "gpqa", "humaneval", "truthfulqa", "ifeval"]

BASE_PROTOCOL = {
    "suite": SUITE_NAME,
    "seed": SEED,
    "apply_chat_template_default": APPLY_CHAT_TEMPLATE,
    "fewshot_as_multiturn_when_chat": True,
    "lm_eval_version": LM_EVAL_VERSION,
    "lm_eval_commit": LM_EVAL_COMMIT,
    "benchmarks": BENCHMARK_PROTOCOLS,
    "human_eval_execution": False,
    "scoring_performed": False,
}


## 6. Stable-file and serialization helpers

JSONL writes are append-only and flushed frequently. On resume, a malformed trailing record
is truncated to the last valid byte; valid records are retained. Existing records from a
different model or protocol cause a hard error rather than being overwritten.


In [6]:
def jsonable(value):
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [jsonable(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, torch.dtype):
        return str(value)
    if hasattr(value, "to_dict"):
        return jsonable(value.to_dict())
    return str(value)


def canonical_json(value):
    return json.dumps(jsonable(value), sort_keys=True, separators=(",", ":"), ensure_ascii=False)


def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(jsonable(payload), indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    os.replace(temporary, path)


def atomic_write_jsonl(path, records):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8", newline="\n") as handle:
        for record in records:
            handle.write(json.dumps(jsonable(record), ensure_ascii=False, separators=(",", ":")) + "\n")
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)


def safe_slug(name):
    slug = re.sub(r"[^A-Za-z0-9._-]+", "_", name.strip()).strip("._")
    if not slug:
        raise ValueError(f"Model name does not produce a safe output directory: {name!r}")
    return slug


def repair_and_read_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    records = []
    valid_offset = 0
    with path.open("rb") as handle:
        while True:
            line = handle.readline()
            if not line:
                break
            try:
                record = json.loads(line.decode("utf-8"))
            except Exception:
                warnings.warn(f"Truncating malformed JSONL tail in {path} at byte {valid_offset}.")
                break
            records.append(record)
            valid_offset = handle.tell()
    if valid_offset != path.stat().st_size:
        with path.open("r+b") as handle:
            handle.truncate(valid_offset)
    seen = set()
    for record in records:
        sample_id = record.get("sample_id")
        if not sample_id or sample_id in seen:
            raise RuntimeError(f"Missing or duplicate sample_id in {path}: {sample_id!r}")
        seen.add(sample_id)
    return records


def append_records(path, records):
    if not records:
        return
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8", newline="\n") as handle:
        for index, record in enumerate(records, start=1):
            handle.write(json.dumps(jsonable(record), ensure_ascii=False, separators=(",", ":")) + "\n")
            if index % FLUSH_EVERY == 0:
                handle.flush()
                os.fsync(handle.fileno())
        handle.flush()
        os.fsync(handle.fileno())


def chunks(items, size):
    for start in range(0, len(items), size):
        yield items[start : start + size]


def display_gpu_memory():
    for gpu_index in range(torch.cuda.device_count()):
        free_bytes, total_bytes = torch.cuda.mem_get_info(gpu_index)
        allocated = torch.cuda.memory_allocated(gpu_index)
        reserved = torch.cuda.memory_reserved(gpu_index)
        print(
            f"GPU {gpu_index} memory: free={free_bytes / 2**30:.2f} GiB, "
            f"allocated={allocated / 2**30:.2f} GiB, reserved={reserved / 2**30:.2f} GiB, "
            f"total={total_bytes / 2**30:.2f} GiB"
        )


## 7. Harness task loading and protocol fingerprint

The exact pinned harness tasks build prompts and inference requests. Dataset revisions are
injected before loading. GPQA's deterministic shuffle is performed once after seeding, and
its shuffled choice text and ground-truth mapping are exported per record. HumanEval uses an
equivalent metric-free harness config because v0.4.9.2's canonical utility executes a
`code_eval` smoke test at import time; that utility is deliberately never imported here.


In [7]:
from lm_eval.api.task import ConfigurableTask
from lm_eval.evaluator_utils import get_task_list
from lm_eval.tasks import TaskManager, get_task_dict


def load_frozen_tasks():
    seed_everything(SEED)
    manager = TaskManager(verbosity="ERROR")
    loaded = {}
    for benchmark in BENCHMARK_ORDER:
        protocol = BENCHMARK_PROTOCOLS[benchmark]
        if benchmark == "humaneval":
            # Do not load the registered `humaneval` YAML: in pinned harness v0.4.9.2 its
            # imported metric utility calls code_eval.compute() immediately. This unregistered
            # task exactly preserves its dataset, prompt, reference, stops, and generation
            # settings while defining no metric and importing no executable-code evaluator.
            task_request = {
                "task": "humaneval_generation_only",
                "dataset_path": protocol["dataset_path"],
                "dataset_kwargs": {"revision": protocol["dataset_revision"]},
                "output_type": "generate_until",
                "test_split": "test",
                "doc_to_text": "{{prompt}}",
                "doc_to_target": "{{test}}\ncheck({{entry_point}})",
                "generation_kwargs": deepcopy(protocol["generation_kwargs"]),
                "repeats": 1,
                "num_fewshot": 0,
                "metric_list": [],
                "unsafe_code": False,
                "metadata": {
                    "version": 1.0,
                    "canonical_harness_task": "humaneval",
                    "generation_only": True,
                    "code_execution": False,
                },
            }
        else:
            task_request = {
                "task": protocol["harness_task"],
                "dataset_kwargs": {"revision": protocol["dataset_revision"]},
                "num_fewshot": protocol["num_fewshot"],
            }
        if benchmark == "humaneval":
            # get_task_dict() pretty-prints by looking up every task in TaskManager's YAML
            # registry. This deliberately unregistered safe task has no YAML entry, so build
            # the same ConfigurableTask directly and bypass only that registry logger.
            task_dict = {
                task_request["task"]: ConfigurableTask(config=task_request)
            }
        else:
            task_dict = get_task_dict([task_request], task_manager=manager)
        outputs = [item for item in get_task_list(task_dict) if item.task is not None]
        if not outputs:
            raise RuntimeError(f"Harness resolved no concrete tasks for {benchmark}.")
        for output in outputs:
            output.task.set_fewshot_seed(SEED)
            output.task._config.num_fewshot = protocol["num_fewshot"]
            if protocol["generation_kwargs"] is not None:
                output.task._config.generation_kwargs = deepcopy(protocol["generation_kwargs"])
        loaded[benchmark] = outputs
        print(f"Loaded {benchmark}: {len(outputs)} concrete harness task(s)")
    return loaded


TASK_OUTPUTS_BY_BENCHMARK = load_frozen_tasks()


def sample_id_for(task_name, doc_id):
    return f"{task_name}:{int(doc_id):06d}"


def expected_inventory(benchmark):
    inventory = []
    for output in TASK_OUTPUTS_BY_BENCHMARK[benchmark]:
        for doc_id, _doc in enumerate(output.task.eval_docs):
            inventory.append(
                {
                    "sample_id": sample_id_for(output.task_name, doc_id),
                    "task_name": output.task_name,
                    "doc_id": int(doc_id),
                }
            )
    return inventory


EXPECTED_INVENTORIES = {name: expected_inventory(name) for name in BENCHMARK_ORDER}
EXPECTED_IDS = {
    name: {item["sample_id"] for item in inventory}
    for name, inventory in EXPECTED_INVENTORIES.items()
}

mmlu_tasks = TASK_OUTPUTS_BY_BENCHMARK["mmlu"]
mmlu_subjects = sorted(item.task_name.removeprefix("mmlu_") for item in mmlu_tasks)
if len(mmlu_tasks) != 57 or len(set(mmlu_subjects)) != 57:
    raise RuntimeError(f"Expected all 57 original MMLU subjects; resolved {len(set(mmlu_subjects))}.")

gpqa_outputs = TASK_OUTPUTS_BY_BENCHMARK["gpqa"]
if len(gpqa_outputs) != 1:
    raise RuntimeError(f"Expected one GPQA Main task, found {len(gpqa_outputs)}.")
gpqa_task = gpqa_outputs[0].task
if gpqa_outputs[0].task_name != "gpqa_main_zeroshot" or gpqa_task.config.dataset_name != "gpqa_main":
    raise RuntimeError("GPQA guard failed: this notebook must use FULL gpqa_main, never gpqa_diamond.")

if len(EXPECTED_IDS["humaneval"]) != 164:
    raise RuntimeError(f"Expected 164 HumanEval problems; found {len(EXPECTED_IDS['humaneval'])}.")


def task_fingerprint(output):
    dataset = output.task.eval_docs
    return {
        "task_name": output.task_name,
        "task_version": (output.task.config.metadata or {}).get("version", output.version),
        "task_config": output.task.config.to_dict(),
        "dataset_fingerprint": getattr(dataset, "_fingerprint", None),
        "sample_count": len(dataset),
    }


RESOLVED_TASKS = {
    benchmark: [task_fingerprint(output) for output in outputs]
    for benchmark, outputs in TASK_OUTPUTS_BY_BENCHMARK.items()
}
FROZEN_PROTOCOL = {
    **BASE_PROTOCOL,
    "resolved_tasks": RESOLVED_TASKS,
    "expected_sample_ids_sha256": {
        name: sha256_text(canonical_json(sorted(ids))) for name, ids in EXPECTED_IDS.items()
    },
}
PROTOCOL_SHA256 = sha256_text(canonical_json(FROZEN_PROTOCOL))
PROTOCOL_PATH = OUTPUT_ROOT_PATH / "protocol.json"
if PROTOCOL_PATH.exists():
    existing_protocol = json.loads(PROTOCOL_PATH.read_text(encoding="utf-8"))
    if existing_protocol.get("protocol_sha256") != PROTOCOL_SHA256:
        raise RuntimeError(
            "Existing OUTPUT_ROOT was created with a different protocol or dataset fingerprint. "
            "Choose a new OUTPUT_ROOT; existing artifacts will not be overwritten."
        )
else:
    atomic_write_json(
        PROTOCOL_PATH,
        {"protocol_sha256": PROTOCOL_SHA256, "protocol": FROZEN_PROTOCOL},
    )

print("Frozen protocol SHA256:", PROTOCOL_SHA256)
for benchmark in BENCHMARK_ORDER:
    print(f"{benchmark:10s}: {len(EXPECTED_IDS[benchmark]):5d} samples")


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

world_religions/test-00000-of-00001.parq(…):   0%|          | 0.00/18.9k [00:00<?, ?B/s]

world_religions/validation-00000-of-0000(…):   0%|          | 0.00/4.94k [00:00<?, ?B/s]

world_religions/dev-00000-of-00001.parqu(…):   0%|          | 0.00/3.30k [00:00<?, ?B/s]

...

Loaded truthfulqa: 1 concrete harness task(s)


README.md: 0.00B [00:00, ?B/s]

ifeval_input_data.jsonl: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Loaded ifeval: 1 concrete harness task(s)
Frozen protocol SHA256: 51f5acedf7c1742f18ed4cdbd614d14dd4fc7a4dbe09783e44f5bd2cbf409301
mmlu      : 14042 samples
gsm8k     :  1319 samples
gpqa      :   448 samples
humaneval :   164 samples
truthfulqa:   817 samples
ifeval    :   541 samples


## 8. Model loader

Each model is resolved from its Kaggle Input, loaded through the harness Hugging Face wrapper
(which uses `AutoTokenizer` and `AutoModelForCausalLM`), and distributed with Accelerate's
automatic device map. No quantization or weight transformation is requested. Flash Attention
is not required. Transformers-v5 `TokenizersBackend` exports are explicitly bridged through
their immutable `tokenizer.json` when the pinned v4 loader cannot resolve the v5 class name;
both tokenizer files are hashed in metadata. The model is put in evaluation mode and gradients
stay disabled.


In [8]:
from lm_eval.models.huggingface import HFLM


def resolve_model_path(configured_path):
    path = Path(configured_path).expanduser().resolve()
    if not path.is_dir():
        raise FileNotFoundError(f"Attached model directory not found: {path}")
    if (path / "config.json").is_file():
        return path
    candidates = sorted({item.parent for item in path.rglob("config.json")})
    if len(candidates) == 1:
        warnings.warn(f"Resolved nested Hugging Face model directory: {candidates[0]}")
        return candidates[0]
    raise FileNotFoundError(
        f"Expected config.json at {path}, or exactly one nested model; found {len(candidates)} candidates."
    )



def load_compatible_tokenizer(resolved_path, model_config):
    """Load local tokenizers, bridging Transformers-v5 exports into pinned Transformers v4."""
    resolved_path = Path(resolved_path)
    trust_remote_code = bool(model_config.get("trust_remote_code", False))
    tokenizer_config_path = resolved_path / "tokenizer_config.json"
    tokenizer_json_path = resolved_path / "tokenizer.json"
    tokenizer_config = {}
    if tokenizer_config_path.is_file():
        tokenizer_config = json.loads(tokenizer_config_path.read_text(encoding="utf-8"))
    declared_class = tokenizer_config.get("tokenizer_class")
    common_kwargs = {
        "trust_remote_code": trust_remote_code,
        "local_files_only": True,
    }
    strategy = "auto_tokenizer"
    compatibility_reason = None
    serialized_probe_ids = None
    try:
        tokenizer = transformers.AutoTokenizer.from_pretrained(
            str(resolved_path),
            use_fast=True,
            **common_kwargs,
        )
    except ValueError as exc:
        is_v5_backend_mismatch = (
            declared_class == "TokenizersBackend"
            and "Tokenizer class TokenizersBackend does not exist" in str(exc)
        )
        if not is_v5_backend_mismatch:
            raise
        if not tokenizer_json_path.is_file():
            raise RuntimeError(
                f"{model_config['name']}: tokenizer declares the Transformers-v5 "
                "TokenizersBackend class, but tokenizer.json is missing; a safe compatibility "
                "load is impossible."
            ) from exc
        warnings.warn(
            f"{model_config['name']}: checkpoint tokenizer was exported by Transformers v5 "
            "as TokenizersBackend. Loading its immutable tokenizer.json through pinned "
            "Transformers v4 PreTrainedTokenizerFast; model files are not modified."
        )
        tokenizer = transformers.PreTrainedTokenizerFast.from_pretrained(
            str(resolved_path),
            tokenizer_file=str(tokenizer_json_path),
            **common_kwargs,
        )
        from tokenizers import Tokenizer as TokenizersLibraryTokenizer

        serialized_probe_ids = TokenizersLibraryTokenizer.from_file(
            str(tokenizer_json_path)
        ).encode("TI-DPO tokenizer compatibility probe", add_special_tokens=False).ids
        strategy = "v5_tokenizersbackend_via_v4_pretrainedtokenizerfast"
        compatibility_reason = "transformers_v5_tokenizer_metadata_loaded_by_pinned_transformers_v4"

    if not isinstance(
        tokenizer,
        (transformers.PreTrainedTokenizer, transformers.PreTrainedTokenizerFast),
    ):
        raise TypeError(f"Unsupported tokenizer object returned: {type(tokenizer).__name__}")
    probe_ids = tokenizer.encode("TI-DPO tokenizer compatibility probe", add_special_tokens=False)
    if not probe_ids or not all(isinstance(token_id, int) for token_id in probe_ids):
        raise RuntimeError(f"{model_config['name']}: tokenizer compatibility probe produced no valid IDs.")
    if serialized_probe_ids is not None and probe_ids != serialized_probe_ids:
        raise RuntimeError(
            f"{model_config['name']}: compatibility tokenizer IDs differ from the immutable "
            "tokenizer.json backend; refusing to run evaluation."
        )

    load_metadata = {
        "strategy": strategy,
        "compatibility_reason": compatibility_reason,
        "declared_tokenizer_class": declared_class,
        "loaded_tokenizer_class": type(tokenizer).__name__,
        "transformers_version": transformers.__version__,
        "tokenizers_version": importlib.metadata.version("tokenizers"),
        "tokenizer_config_sha256": (
            sha256_file(tokenizer_config_path) if tokenizer_config_path.is_file() else None
        ),
        "tokenizer_json_sha256": sha256_file(tokenizer_json_path) if tokenizer_json_path.is_file() else None,
        "probe_token_count": len(probe_ids),
        "probe_matches_serialized_tokenizer_json": (
            probe_ids == serialized_probe_ids if serialized_probe_ids is not None else None
        ),
        "input_files_modified": False,
    }
    print(
        f"Tokenizer for {model_config['name']}: {type(tokenizer).__name__} "
        f"(strategy={strategy}, declared={declared_class!r})"
    )
    return tokenizer, load_metadata

def select_chat_template(lm, model_config):
    requested = bool(model_config.get("apply_chat_template", APPLY_CHAT_TEMPLATE))
    raw_template = getattr(lm.tokenizer, "chat_template", None)
    selector = model_config.get("chat_template", True)
    if not requested:
        return False, None, raw_template, "disabled_by_configuration"
    if not raw_template:
        warnings.warn(
            f"{model_config['name']}: tokenizer has no official chat template; "
            "chat templating is explicitly disabled for this model. No fallback template is invented."
        )
        return False, None, raw_template, "missing_official_tokenizer_template"
    if isinstance(raw_template, dict) and selector is True and "default" not in raw_template:
        warnings.warn(
            f"{model_config['name']}: tokenizer exposes multiple templates without a default. "
            "Set the optional per-model 'chat_template' name to apply one; templating is disabled for now."
        )
        return False, None, raw_template, "multiple_templates_without_selected_default"
    selected = lm.chat_template(selector)
    if not selected:
        warnings.warn(f"{model_config['name']}: harness could not select an official chat template.")
        return False, None, raw_template, "harness_returned_no_template"
    # Make named/dict selection explicit for tokenizer.apply_chat_template.
    lm.tokenizer.chat_template = selected
    return True, selected, raw_template, "official_tokenizer_template"


def load_model(model_config):
    resolved_path = resolve_model_path(model_config["path"])
    batch_size = model_config.get("batch_size", DEFAULT_BATCH_SIZE)
    max_batch_size = int(model_config.get("max_batch_size", DEFAULT_MAX_BATCH_SIZE))
    print(f"Loading {model_config['name']} from {resolved_path}")
    seed_everything(SEED)
    tokenizer, tokenizer_load_metadata = load_compatible_tokenizer(resolved_path, model_config)
    lm = HFLM(
        pretrained=str(resolved_path),
        tokenizer=tokenizer,
        backend="causal",
        revision="main",
        device="cuda",
        dtype=MODEL_DTYPE,
        batch_size=batch_size,
        max_batch_size=max_batch_size,
        parallelize=True,
        trust_remote_code=bool(model_config.get("trust_remote_code", False)),
        use_fast_tokenizer=True,
    )
    lm.model.eval()
    torch.set_grad_enabled(False)
    chat_applied, selected_template, raw_template, template_status = select_chat_template(lm, model_config)
    return lm, resolved_path, {
        "requested": bool(model_config.get("apply_chat_template", APPLY_CHAT_TEMPLATE)),
        "applied": chat_applied,
        "status": template_status,
        "selected_template": selected_template,
        "raw_tokenizer_chat_template": jsonable(raw_template),
        "sha256": sha256_text(selected_template) if selected_template else None,
        "tokenizer_loading": tokenizer_load_metadata,
    }


## 9. Multiple-choice inference and stable records

MMLU, GPQA Main, and TruthfulQA MC2 are evaluated as conditional-likelihood requests.
Every choice receives its unrounded total log-likelihood and greedy-match flag. Choice text,
exact continuation, ordering, character/byte lengths, labels, and references are preserved.
No answer letter is generated and no accuracy is calculated.


In [9]:
def choice_texts_for(benchmark, task, doc):
    if benchmark == "mmlu":
        return list(doc["choices"])
    if benchmark == "gpqa":
        return [doc[f"choice{index}"] for index in range(1, 5)]
    if benchmark == "truthfulqa":
        return list(doc["mc2_targets"]["choices"])
    raise KeyError(benchmark)


def reference_for(benchmark, task, doc, labels, texts):
    if benchmark == "mmlu":
        target_index = int(doc["answer"])
        return {
            "ground_truth_index": target_index,
            "ground_truth_label": labels[target_index],
            "ground_truth_text": texts[target_index],
        }
    if benchmark == "gpqa":
        target_label = str(task.doc_to_target(doc))
        target_index = labels.index(target_label)
        return {
            "ground_truth_index": target_index,
            "ground_truth_label": target_label,
            "ground_truth_text": texts[target_index],
            "original_correct_answer": doc.get("Correct Answer"),
            "shuffled_choice_mapping": [
                {
                    "index": index,
                    "label": labels[index],
                    "text": text,
                    "is_correct": index == target_index,
                }
                for index, text in enumerate(texts)
            ],
        }
    if benchmark == "truthfulqa":
        truth_labels = [int(item) for item in doc["mc2_targets"]["labels"]]
        return {
            "candidate_truth_labels": truth_labels,
            "true_choice_indices": [i for i, value in enumerate(truth_labels) if value == 1],
            "false_choice_indices": [i for i, value in enumerate(truth_labels) if value == 0],
            "mc2_normalization": "softmax over all saved raw candidate log-likelihoods",
        }
    raise KeyError(benchmark)


def build_common_record(model_context, benchmark, output, doc_id, prompt, reference, prediction, request_config):
    task = output.task
    task_version = (task.config.metadata or {}).get("version", output.version)
    subject = output.task_name.removeprefix("mmlu_") if benchmark == "mmlu" else None
    source_dataset = task.eval_docs
    return {
        "suite": SUITE_NAME,
        "protocol_sha256": PROTOCOL_SHA256,
        "model_name": model_context["model_name"],
        "model_path": model_context["model_path"],
        "benchmark": benchmark,
        "task_name": output.task_name,
        "task_version": task_version,
        "sample_id": sample_id_for(output.task_name, doc_id),
        "subject": subject,
        "num_fewshot": BENCHMARK_PROTOCOLS[benchmark]["num_fewshot"],
        "prompt": prompt,
        "reference": jsonable(reference),
        "prediction": jsonable(prediction),
        "generation_config": jsonable(request_config),
        "metadata": {
            "doc_id": int(doc_id),
            "source_document": jsonable(task.eval_docs[int(doc_id)]),
            "lm_eval_task_config": task.config.to_dict(),
            "dataset_fingerprint": getattr(source_dataset, "_fingerprint", None),
            "dataset_revision": BENCHMARK_PROTOCOLS[benchmark]["dataset_revision"],
            "apply_chat_template": model_context["chat_template"]["applied"],
            "chat_template_sha256": model_context["chat_template"]["sha256"],
            "fewshot_as_multiturn": model_context["chat_template"]["applied"],
            "lm_eval_version": LM_EVAL_VERSION,
            "lm_eval_commit": LM_EVAL_COMMIT,
            "created_at": utc_now(),
        },
    }


def build_mc_record(model_context, benchmark, output, doc_id, instances, responses):
    task = output.task
    doc = instances[0].doc
    labels = [str(item) for item in task.doc_to_choice(doc)]
    texts = choice_texts_for(benchmark, task, doc)
    if not (len(labels) == len(texts) == len(instances) == len(responses)):
        raise RuntimeError(f"Choice/request cardinality mismatch for {output.task_name}:{doc_id}")
    prompt = instances[0].args[0]
    if any(instance.args[0] != prompt for instance in instances):
        raise RuntimeError("Expected one shared multiple-choice prompt per document.")
    prediction_choices = []
    for index, (label, text, instance, response) in enumerate(zip(labels, texts, instances, responses)):
        loglikelihood, is_greedy = response
        continuation = instance.args[1]
        prediction_choices.append(
            {
                "index": index,
                "label": label,
                "text": text,
                "continuation": continuation,
                "loglikelihood": float(loglikelihood),
                "is_greedy": bool(is_greedy),
                "harness_choice_char_length": len(label),
                "choice_text_char_length": len(text),
                "choice_text_byte_length": len(text.encode("utf-8")),
                "continuation_char_length": len(continuation),
            }
        )
    return build_common_record(
        model_context=model_context,
        benchmark=benchmark,
        output=output,
        doc_id=doc_id,
        prompt=prompt,
        reference=reference_for(benchmark, task, doc, labels, texts),
        prediction={"choices": prediction_choices},
        request_config={
            "request_type": "loglikelihood",
            "target_delimiter": task.config.target_delimiter,
            "choice_order": labels,
        },
    )


## 10. Generation inference and stable records

GSM8K, HumanEval, and IFEval use deterministic `generate_until` requests. The returned model
continuation is stored unchanged as `prediction.raw_text` together with the exact request
parameters. IFEval's untemplated source prompt is separately retained even when the model's
official chat template changes the rendered model input.


In [10]:
def generation_reference(benchmark, doc):
    if benchmark == "gsm8k":
        return {"answer": doc.get("answer"), "question": doc.get("question")}
    if benchmark == "humaneval":
        return {
            "task_id": doc.get("task_id"),
            "entry_point": doc.get("entry_point"),
            "test": doc.get("test"),
            "canonical_solution": doc.get("canonical_solution"),
            "execution_performed": False,
        }
    if benchmark == "ifeval":
        return {
            "original_prompt": doc.get("prompt"),
            "key": doc.get("key"),
            "instruction_id_list": doc.get("instruction_id_list"),
            "kwargs": doc.get("kwargs"),
        }
    raise KeyError(benchmark)


def build_generation_record(model_context, benchmark, output, doc_id, instance, response):
    doc = instance.doc
    prompt, request_kwargs = instance.args
    frozen_kwargs = BENCHMARK_PROTOCOLS[benchmark]["generation_kwargs"]
    if request_kwargs != frozen_kwargs:
        raise RuntimeError(
            f"Harness generation kwargs drifted for {benchmark}: {request_kwargs!r} != {frozen_kwargs!r}"
        )
    return build_common_record(
        model_context=model_context,
        benchmark=benchmark,
        output=output,
        doc_id=doc_id,
        prompt=prompt,
        reference=generation_reference(benchmark, doc),
        prediction={"raw_text": response},
        request_config={
            **request_kwargs,
            "request_type": "generate_until",
            "num_return_sequences": 1,
            "harness_stop_sequences_removed_from_returned_text": bool(request_kwargs.get("until")),
        },
    )


## 11. Resume, integrity, and benchmark runners

A benchmark is skipped only when `DONE`, its exact expected sample-ID set, JSONL integrity,
protocol hash, and file SHA256 all agree. Partial files resume at missing sample IDs. The
runner writes after small chunks and never invokes any task's scoring function.


In [11]:
def benchmark_paths(model_dir, benchmark):
    directory = Path(model_dir) / benchmark
    return {
        "dir": directory,
        "samples": directory / "samples.jsonl",
        "done": directory / "DONE",
        "incomplete": directory / "INCOMPLETE.json",
    }


def validate_records(model_context, benchmark, records):
    ids = [record.get("sample_id") for record in records]
    id_set = set(ids)
    expected = EXPECTED_IDS[benchmark]
    unknown = id_set - expected
    if unknown:
        raise RuntimeError(f"{benchmark} contains {len(unknown)} unknown sample IDs; refusing to overwrite.")
    for record in records:
        if record.get("suite") != SUITE_NAME or record.get("protocol_sha256") != PROTOCOL_SHA256:
            raise RuntimeError(f"{benchmark} contains records from a different suite/protocol.")
        if record.get("model_name") != model_context["model_name"]:
            raise RuntimeError(f"{benchmark} contains records from a different model name.")
        if Path(record.get("model_path", "")).resolve() != Path(model_context["model_path"]).resolve():
            raise RuntimeError(f"{benchmark} contains records from a different model path.")
        prediction = record.get("prediction", {})
        if BENCHMARK_PROTOCOLS[benchmark]["output_type"] == "multiple_choice":
            choices = prediction.get("choices")
            if not choices or any("loglikelihood" not in choice for choice in choices):
                raise RuntimeError(f"{benchmark} has an incomplete likelihood record: {record.get('sample_id')}")
        elif "raw_text" not in prediction:
            raise RuntimeError(f"{benchmark} has an incomplete generation: {record.get('sample_id')}")
    return {
        "valid": id_set == expected and len(ids) == len(expected),
        "samples": len(records),
        "expected": len(expected),
        "missing": len(expected - id_set),
    }


def benchmark_is_complete(model_context, benchmark):
    paths = benchmark_paths(model_context["model_dir"], benchmark)
    if not paths["done"].is_file() or not paths["samples"].is_file():
        return False, None
    records = repair_and_read_jsonl(paths["samples"])
    validation = validate_records(model_context, benchmark, records)
    if not validation["valid"]:
        return False, validation
    done = json.loads(paths["done"].read_text(encoding="utf-8"))
    digest = sha256_file(paths["samples"])
    valid_done = (
        done.get("protocol_sha256") == PROTOCOL_SHA256
        and done.get("samples") == len(records)
        and done.get("sha256") == digest
    )
    return valid_done, {**validation, "sha256": digest}


def prepare_requests(output, model_context):
    task = output.task
    task.set_fewshot_seed(SEED)
    task._config.num_fewshot = BENCHMARK_PROTOCOLS[model_context["benchmark"]]["num_fewshot"]
    generation_kwargs = BENCHMARK_PROTOCOLS[model_context["benchmark"]]["generation_kwargs"]
    if generation_kwargs is not None:
        task._config.generation_kwargs = deepcopy(generation_kwargs)
    apply_chat = model_context["chat_template"]["applied"]
    task.build_all_requests(
        rank=0,
        world_size=1,
        cache_requests=False,
        rewrite_requests_cache=False,
        system_instruction=None,
        apply_chat_template=apply_chat,
        fewshot_as_multiturn=apply_chat,
        chat_template=model_context["lm"].apply_chat_template if apply_chat else None,
        tokenizer_name=model_context["lm"].tokenizer_name if apply_chat else "",
    )
    grouped = defaultdict(list)
    for instance in task.instances:
        grouped[int(instance.doc_id)].append(instance)
    return [(doc_id, grouped[doc_id]) for doc_id in sorted(grouped)]


@torch.inference_mode()
def run_benchmark(model_context, benchmark):
    model_context = {**model_context, "benchmark": benchmark}
    paths = benchmark_paths(model_context["model_dir"], benchmark)
    paths["dir"].mkdir(parents=True, exist_ok=True)
    complete, validation = benchmark_is_complete(model_context, benchmark)
    if complete:
        print(f"SKIP {model_context['model_name']} / {benchmark}: DONE passed integrity checks.")
        return validation

    records = repair_and_read_jsonl(paths["samples"])
    validation = validate_records(model_context, benchmark, records)
    completed_ids = {record["sample_id"] for record in records}
    print(
        f"RUN  {model_context['model_name']} / {benchmark}: "
        f"{len(completed_ids)}/{len(EXPECTED_IDS[benchmark])} already complete"
    )
    display_gpu_memory()
    if paths["incomplete"].exists():
        paths["incomplete"].unlink()

    benchmark_progress = tqdm(
        total=len(EXPECTED_IDS[benchmark]),
        initial=len(completed_ids),
        desc=f"{model_context['model_name']} | {benchmark}",
        unit="sample",
    )
    try:
        for output in TASK_OUTPUTS_BY_BENCHMARK[benchmark]:
            request_groups = prepare_requests(output, model_context)
            pending = [
                (doc_id, instances)
                for doc_id, instances in request_groups
                if sample_id_for(output.task_name, doc_id) not in completed_ids
            ]
            for group_chunk in chunks(pending, SAMPLE_CHUNK_SIZE):
                if BENCHMARK_PROTOCOLS[benchmark]["output_type"] == "multiple_choice":
                    flat_requests = [instance for _doc_id, group in group_chunk for instance in group]
                    flat_responses = model_context["lm"].loglikelihood(flat_requests, disable_tqdm=True)
                    new_records = []
                    offset = 0
                    for doc_id, instances in group_chunk:
                        count = len(instances)
                        responses = flat_responses[offset : offset + count]
                        offset += count
                        new_records.append(
                            build_mc_record(model_context, benchmark, output, doc_id, instances, responses)
                        )
                else:
                    flat_requests = [group[0] for _doc_id, group in group_chunk]
                    if any(len(group) != 1 for _doc_id, group in group_chunk):
                        raise RuntimeError(f"Expected one generation request per {benchmark} sample.")
                    responses = model_context["lm"].generate_until(flat_requests, disable_tqdm=True)
                    new_records = [
                        build_generation_record(model_context, benchmark, output, doc_id, group[0], response)
                        for (doc_id, group), response in zip(group_chunk, responses)
                    ]
                append_records(paths["samples"], new_records)
                completed_ids.update(record["sample_id"] for record in new_records)
                benchmark_progress.update(len(new_records))
            output.task._instances = []
    finally:
        benchmark_progress.close()

    records = repair_and_read_jsonl(paths["samples"])
    validation = validate_records(model_context, benchmark, records)
    if not validation["valid"]:
        atomic_write_json(
            paths["incomplete"],
            {**validation, "protocol_sha256": PROTOCOL_SHA256, "updated_at": utc_now()},
        )
        raise RuntimeError(f"{benchmark} is incomplete after generation: {validation}")

    digest = sha256_file(paths["samples"])
    done_payload = {
        "status": "complete",
        "benchmark": benchmark,
        "samples": len(records),
        "expected_samples": len(EXPECTED_IDS[benchmark]),
        "sha256": digest,
        "protocol_sha256": PROTOCOL_SHA256,
        "completed_at": utc_now(),
    }
    atomic_write_json(paths["done"], done_payload)
    return {**validation, "sha256": digest}


## 12. Model metadata and main sequential loop

Models run one at a time. Results are committed benchmark-by-benchmark. After each model,
model/tokenizer objects are deleted, garbage collection runs, and every CUDA cache is emptied
before the next checkpoint is loaded.


In [12]:
PIP_FREEZE = subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"], text=True
).splitlines()


def model_metadata(model_config, model_context):
    lm = model_context["lm"]
    model = lm.model
    config_dict = lm.config.to_dict() if hasattr(lm.config, "to_dict") else jsonable(lm.config)
    parameter_count = model.num_parameters() if hasattr(model, "num_parameters") else sum(
        parameter.numel() for parameter in model.parameters()
    )
    return {
        "suite": SUITE_NAME,
        "protocol_sha256": PROTOCOL_SHA256,
        "model_display_name": model_config["name"],
        "configured_path": model_config["path"],
        "absolute_kaggle_path": model_context["model_path"],
        "model_config": config_dict,
        "architecture": config_dict.get("architectures", []),
        "parameter_count": int(parameter_count),
        "dtype": str(getattr(model, "dtype", MODEL_DTYPE)),
        "hf_device_map": jsonable(getattr(model, "hf_device_map", None)),
        "tokenizer_name_or_path": str(getattr(lm.tokenizer, "name_or_path", model_context["model_path"])),
        "tokenizer_class": type(lm.tokenizer).__name__,
        "tokenizer_loading": model_context["chat_template"].get("tokenizer_loading"),
        "chat_template": model_context["chat_template"],
        "package_versions": PACKAGE_VERSIONS,
        "pip_freeze": PIP_FREEZE,
        "gpu": GPU_INFO,
        "seed": SEED,
        "suite_version": SUITE_NAME,
        "benchmark_configs": BENCHMARK_PROTOCOLS,
        "resolved_harness_tasks": RESOLVED_TASKS,
        "generation_kwargs": {
            name: config["generation_kwargs"]
            for name, config in BENCHMARK_PROTOCOLS.items()
            if config["generation_kwargs"] is not None
        },
        "lm_eval_version": LM_EVAL_VERSION,
        "lm_eval_commit": LM_EVAL_COMMIT,
        "created_at": utc_now(),
        "scoring_performed": False,
        "human_eval_code_executed": False,
    }


def write_or_validate_model_metadata(path, payload):
    path = Path(path)
    if path.exists():
        existing = json.loads(path.read_text(encoding="utf-8"))
        identity_fields = [
            "protocol_sha256",
            "model_display_name",
            "absolute_kaggle_path",
            "lm_eval_commit",
        ]
        mismatches = [field for field in identity_fields if existing.get(field) != payload.get(field)]
        existing_chat_hash = existing.get("chat_template", {}).get("sha256")
        new_chat_hash = payload.get("chat_template", {}).get("sha256")
        if existing_chat_hash != new_chat_hash:
            mismatches.append("chat_template.sha256")
        if mismatches:
            raise RuntimeError(
                f"Existing metadata differs in {mismatches}; choose a new model name/OUTPUT_ROOT."
            )
        return existing
    atomic_write_json(path, payload)
    return payload


MANIFEST_PATH = OUTPUT_ROOT_PATH / "manifest.json"


def load_manifest():
    if MANIFEST_PATH.exists():
        manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
        if manifest.get("protocol_sha256") != PROTOCOL_SHA256:
            raise RuntimeError("Existing manifest belongs to a different frozen protocol.")
        return manifest
    return {
        "suite": SUITE_NAME,
        "protocol_sha256": PROTOCOL_SHA256,
        "created_at": utc_now(),
        "models": [],
        "entries": [],
    }


MANIFEST = load_manifest()


def update_manifest(model_name, benchmark, status, samples, digest, file_path, error=None):
    MANIFEST["entries"] = [
        entry
        for entry in MANIFEST["entries"]
        if not (entry.get("model") == model_name and entry.get("benchmark") == benchmark)
    ]
    entry = {
        "model": model_name,
        "benchmark": benchmark,
        "status": status,
        "samples": int(samples),
        "sha256": digest,
        "file": str(Path(file_path).relative_to(OUTPUT_ROOT_PATH)),
    }
    if error:
        entry["error"] = error
    MANIFEST["entries"].append(entry)
    MANIFEST["entries"].sort(key=lambda item: (item["model"], BENCHMARK_ORDER.index(item["benchmark"])))
    MANIFEST["updated_at"] = utc_now()
    atomic_write_json(MANIFEST_PATH, MANIFEST)


def validate_model_configuration():
    if not MODELS:
        raise ValueError("MODELS is empty. Add at least one attached Hugging Face model in section 1.")
    seen = set()
    for item in MODELS:
        if not isinstance(item, dict) or not item.get("name") or not item.get("path"):
            raise ValueError(f"Each MODELS entry needs non-empty name/path fields: {item!r}")
        if "<MODEL_" in item["name"] or "<MODEL_" in item["path"]:
            raise ValueError("Replace the <MODEL_NAME_...>/<MODEL_PATH_...> placeholders in section 1.")
        slug = safe_slug(item["name"])
        if slug in seen:
            raise ValueError(f"Two model names map to the same output directory: {slug}")
        seen.add(slug)


validate_model_configuration()
for configured_model in MODELS:
    model_name = configured_model["name"]
    model_slug = safe_slug(model_name)
    model_dir = OUTPUT_ROOT_PATH / model_slug
    model_dir.mkdir(parents=True, exist_ok=True)
    resolved_model_path = resolve_model_path(configured_model["path"])
    pre_context = {
        "model_name": model_name,
        "model_path": str(resolved_model_path),
        "model_dir": str(model_dir),
    }
    if model_name not in MANIFEST["models"]:
        MANIFEST["models"].append(model_name)
        atomic_write_json(MANIFEST_PATH, MANIFEST)

    already_complete = True
    for benchmark in BENCHMARK_ORDER:
        done, info = benchmark_is_complete(pre_context, benchmark)
        if not done:
            already_complete = False
            break
    if already_complete:
        print(f"SKIP MODEL {model_name}: every benchmark passed DONE integrity checks.")
        if model_name not in MANIFEST["models"]:
            MANIFEST["models"].append(model_name)
        for benchmark in BENCHMARK_ORDER:
            paths = benchmark_paths(model_dir, benchmark)
            records = repair_and_read_jsonl(paths["samples"])
            update_manifest(
                model_name,
                benchmark,
                "complete",
                len(records),
                sha256_file(paths["samples"]),
                paths["samples"],
            )
        continue

    lm = None
    try:
        lm, resolved_model_path, chat_metadata = load_model(configured_model)
        context = {
            "lm": lm,
            "model_name": model_name,
            "model_path": str(resolved_model_path),
            "model_dir": str(model_dir),
            "chat_template": chat_metadata,
        }
        metadata = model_metadata(configured_model, context)
        write_or_validate_model_metadata(model_dir / "metadata.json", metadata)
        for benchmark in BENCHMARK_ORDER:
            paths = benchmark_paths(model_dir, benchmark)
            try:
                result = run_benchmark(context, benchmark)
                update_manifest(
                    model_name,
                    benchmark,
                    "complete",
                    result["samples"],
                    result["sha256"],
                    paths["samples"],
                )
            except Exception as exc:
                records = repair_and_read_jsonl(paths["samples"])
                error_payload = {
                    "status": "incomplete",
                    "error_type": type(exc).__name__,
                    "error": str(exc),
                    "samples": len(records),
                    "expected_samples": len(EXPECTED_IDS[benchmark]),
                    "updated_at": utc_now(),
                }
                atomic_write_json(paths["incomplete"], error_payload)
                update_manifest(
                    model_name,
                    benchmark,
                    "incomplete",
                    len(records),
                    sha256_file(paths["samples"]) if paths["samples"].exists() else None,
                    paths["samples"],
                    error=f"{type(exc).__name__}: {exc}",
                )
                print(f"ERROR {model_name} / {benchmark}: {type(exc).__name__}: {exc}")
                traceback.print_exc()
    except Exception as exc:
        print(f"MODEL ERROR {model_name}: {type(exc).__name__}: {exc}")
        traceback.print_exc()
        for benchmark in BENCHMARK_ORDER:
            paths = benchmark_paths(model_dir, benchmark)
            done, info = benchmark_is_complete(pre_context, benchmark)
            if done:
                continue
            records = repair_and_read_jsonl(paths["samples"])
            paths["dir"].mkdir(parents=True, exist_ok=True)
            error_payload = {
                "status": "incomplete",
                "error_type": type(exc).__name__,
                "error": str(exc),
                "samples": len(records),
                "expected_samples": len(EXPECTED_IDS[benchmark]),
                "updated_at": utc_now(),
            }
            atomic_write_json(paths["incomplete"], error_payload)
            update_manifest(
                model_name,
                benchmark,
                "incomplete",
                len(records),
                sha256_file(paths["samples"]) if paths["samples"].exists() else None,
                paths["samples"],
                error=f"{type(exc).__name__}: {exc}",
            )
    finally:
        if lm is not None:
            try:
                del lm._model
            except Exception:
                pass
            del lm
        gc.collect()
        torch.cuda.empty_cache()
        for gpu_index in range(torch.cuda.device_count()):
            with torch.cuda.device(gpu_index):
                torch.cuda.empty_cache()
        print(f"Released model resources for {model_name}.")


Loading VPDPO_B_Norm_DPO from /kaggle/input/datasets/lonnyng/b-norm-dpo-olmo-1b/olmo2_bees_b_norm_dpo/run/final
MODEL ERROR VPDPO_B_Norm_DPO: ValueError: Tokenizer class TokenizersBackend does not exist or is not currently imported.


Traceback (most recent call last):
  File "/tmp/ipykernel_58/48561468.py", line 170, in <cell line: 0>
    lm, resolved_model_path, chat_metadata = load_model(configured_model)
                                             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_58/2594912500.py", line 52, in load_model
    lm = HFLM(
         ^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/models/huggingface.py", line 201, in __init__
    self._create_tokenizer(
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/models/huggingface.py", line 771, in _create_tokenizer
    self.tokenizer = transformers.AutoTokenizer.from_pretrained(
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/auto/tokenization_auto.py", line 1137, in from_pretrained
    raise ValueError(
ValueError: Tokenizer class TokenizersBackend does not exist or is not currently imported.


Released model resources for VPDPO_B_Norm_DPO.
Loading VPDPO_B_Norm_VDPO from /kaggle/input/datasets/ahmadsubhaniiqbal/b-norm-vpdpo-olmo-1b/olmo2_bees_b_norm_vdpo/run/final
MODEL ERROR VPDPO_B_Norm_VDPO: ValueError: Tokenizer class TokenizersBackend does not exist or is not currently imported.


Traceback (most recent call last):
  File "/tmp/ipykernel_58/48561468.py", line 170, in <cell line: 0>
    lm, resolved_model_path, chat_metadata = load_model(configured_model)
                                             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_58/2594912500.py", line 52, in load_model
    lm = HFLM(
         ^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/models/huggingface.py", line 201, in __init__
    self._create_tokenizer(
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/models/huggingface.py", line 771, in _create_tokenizer
    self.tokenizer = transformers.AutoTokenizer.from_pretrained(
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/auto/tokenization_auto.py", line 1137, in from_pretrained
    raise ValueError(
ValueError: Tokenizer class TokenizersBackend does not exist or is not currently imported.


Released model resources for VPDPO_B_Norm_VDPO.
Loading VPDPO_B_DPO from /kaggle/input/datasets/ahmadsubhaniiqbal/olmo-1b-b-d
MODEL ERROR VPDPO_B_DPO: ValueError: Tokenizer class TokenizersBackend does not exist or is not currently imported.


...

Released model resources for SimPO.
Loading SAMPO from /kaggle/input/datasets/lonnyng/sampo-models/olmo2_bees_sampo/run/final
MODEL ERROR SAMPO: ValueError: Tokenizer class TokenizersBackend does not exist or is not currently imported.


Traceback (most recent call last):
  File "/tmp/ipykernel_58/48561468.py", line 170, in <cell line: 0>
    lm, resolved_model_path, chat_metadata = load_model(configured_model)
                                             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_58/2594912500.py", line 52, in load_model
    lm = HFLM(
         ^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/models/huggingface.py", line 201, in __init__
    self._create_tokenizer(
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/models/huggingface.py", line 771, in _create_tokenizer
    self.tokenizer = transformers.AutoTokenizer.from_pretrained(
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/auto/tokenization_auto.py", line 1137, in from_pretrained
    raise ValueError(
ValueError: Tokenizer class TokenizersBackend does not exist or is not currently imported.


Released model resources for SAMPO.
Loading TIDPO from /kaggle/input/datasets/ahmadsubhaniiqbal/olmo-bees-tidpo/olmo2_bees_tidpo/run/final
MODEL ERROR TIDPO: ValueError: Tokenizer class TokenizersBackend does not exist or is not currently imported.


Traceback (most recent call last):
  File "/tmp/ipykernel_58/48561468.py", line 170, in <cell line: 0>
    lm, resolved_model_path, chat_metadata = load_model(configured_model)
                                             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_58/2594912500.py", line 52, in load_model
    lm = HFLM(
         ^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/models/huggingface.py", line 201, in __init__
    self._create_tokenizer(
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/models/huggingface.py", line 771, in _create_tokenizer
    self.tokenizer = transformers.AutoTokenizer.from_pretrained(
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/auto/tokenization_auto.py", line 1137, in from_pretrained
    raise ValueError(
ValueError: Tokenizer class TokenizersBackend does not exist or is not currently imported.


Released model resources for TIDPO.


## 13. Final validation

This pass re-reads every JSONL file, checks the exact sample-ID inventory and raw prediction
fields, verifies `DONE` hashes, and refreshes the manifest. Incomplete benchmarks remain
clearly marked and can be resumed by rerunning the notebook with the same outputs attached or
restored under `OUTPUT_ROOT`.


In [13]:
FINAL_VALIDATION = []
for configured_model in MODELS:
    model_name = configured_model["name"]
    model_dir = OUTPUT_ROOT_PATH / safe_slug(model_name)
    resolved_model_path = resolve_model_path(configured_model["path"])
    context = {
        "model_name": model_name,
        "model_path": str(resolved_model_path),
        "model_dir": str(model_dir),
    }
    for benchmark in BENCHMARK_ORDER:
        paths = benchmark_paths(model_dir, benchmark)
        complete, info = benchmark_is_complete(context, benchmark)
        row = {
            "model": model_name,
            "benchmark": benchmark,
            "status": "complete" if complete else "incomplete",
            "samples": (info or {}).get("samples", 0),
            "expected": len(EXPECTED_IDS[benchmark]),
        }
        FINAL_VALIDATION.append(row)
        print(
            f"{row['model']} | {row['benchmark']}: {row['status']} "
            f"({row['samples']}/{row['expected']})"
        )
atomic_write_json(OUTPUT_ROOT_PATH / "validation.json", FINAL_VALIDATION)


VPDPO_B_Norm_DPO | mmlu: incomplete (0/14042)
VPDPO_B_Norm_DPO | gsm8k: incomplete (0/1319)
VPDPO_B_Norm_DPO | gpqa: incomplete (0/448)
VPDPO_B_Norm_DPO | humaneval: incomplete (0/164)
VPDPO_B_Norm_DPO | truthfulqa: incomplete (0/817)
VPDPO_B_Norm_DPO | ifeval: incomplete (0/541)
VPDPO_B_Norm_VDPO | mmlu: incomplete (0/14042)
VPDPO_B_Norm_VDPO | gsm8k: incomplete (0/1319)
VPDPO_B_Norm_VDPO | gpqa: incomplete (0/448)
VPDPO_B_Norm_VDPO | humaneval: incomplete (0/164)
VPDPO_B_Norm_VDPO | truthfulqa: incomplete (0/817)
VPDPO_B_Norm_VDPO | ifeval: incomplete (0/541)
VPDPO_B_DPO | mmlu: incomplete (0/14042)
VPDPO_B_DPO | gsm8k: incomplete (0/1319)
VPDPO_B_DPO | gpqa: incomplete (0/448)
VPDPO_B_DPO | humaneval: incomplete (0/164)
VPDPO_B_DPO | truthfulqa: incomplete (0/817)
VPDPO_B_DPO | ifeval: incomplete (0/541)
VPDPO_B_VDPO | mmlu: incomplete (0/14042)
VPDPO_B_VDPO | gsm8k: incomplete (0/1319)
VPDPO_B_VDPO | gpqa: incomplete (0/448)
VPDPO_B_VDPO | humaneval: incomplete (0/164)
VPDPO_B_VDPO

## 14. Frozen local scoring protocol

Scoring reads only the completed frozen JSONL files; no model is loaded again. The pinned
harness task implementations supply benchmark semantics, and this independent scoring
protocol receives its own hash. No judge, network model, or external inference API is used.


In [14]:
EVALUATION_SUITE_NAME = "tidpo_local_scoring_v1"
if int(HUMANEVAL_NUM_WORKERS) < 1 or float(HUMANEVAL_TIMEOUT_SECONDS) <= 0:
    raise ValueError("HumanEval workers and timeout must both be positive.")
EVALUATION_PROTOCOL = {
    "name": EVALUATION_SUITE_NAME,
    "source_generation_protocol_sha256": PROTOCOL_SHA256,
    "lm_eval_version": LM_EVAL_VERSION,
    "lm_eval_commit": LM_EVAL_COMMIT,
    "code_eval_revision": CODE_EVAL_REVISION,
    "metrics": {
        "mmlu": {
            "primary": "accuracy",
            "rule": "argmax over all saved candidate conditional log-likelihoods",
            "aggregation": "sample-weighted mean over all 57 subjects",
        },
        "gsm8k": {
            "primary": ["strict_exact_match", "flexible_exact_match"],
            "rule": "pinned lm-eval v0.4.9.2 GSM8K regex filters and exact-match normalization",
        },
        "gpqa": {
            "primary": ["accuracy", "length_normalized_accuracy"],
            "rule": "pinned gpqa_main_zeroshot ConfigurableTask.process_results",
        },
        "humaneval": {
            "primary": "pass_at_1",
            "rule": "official HumanEval tests via pinned evaluate code_eval",
            "num_workers": HUMANEVAL_NUM_WORKERS,
            "timeout_seconds": HUMANEVAL_TIMEOUT_SECONDS,
            "one_completion_per_problem": True,
        },
        "truthfulqa": {
            "primary": "mc2_accuracy",
            "rule": "normalized probability mass of the true answer set from every saved log-likelihood",
        },
        "ifeval": {
            "primary": [
                "prompt_level_strict_accuracy",
                "instruction_level_strict_accuracy",
                "prompt_level_loose_accuracy",
                "instruction_level_loose_accuracy",
            ],
            "rule": "pinned original IFEval instruction checks",
        },
    },
    "no_llm_judge": True,
    "no_model_rerun_for_scoring": True,
}
SCORING_PROTOCOL_SHA256 = sha256_text(canonical_json(EVALUATION_PROTOCOL))
SCORING_PROTOCOL_PATH = OUTPUT_ROOT_PATH / "scoring_protocol.json"
scoring_protocol_payload = {
    "scoring_protocol_sha256": SCORING_PROTOCOL_SHA256,
    "protocol": EVALUATION_PROTOCOL,
}
if SCORING_PROTOCOL_PATH.exists():
    existing_scoring_protocol = json.loads(SCORING_PROTOCOL_PATH.read_text(encoding="utf-8"))
    if existing_scoring_protocol.get("scoring_protocol_sha256") != SCORING_PROTOCOL_SHA256:
        raise RuntimeError(
            "Existing scores use a different scoring protocol. Preserve OUTPUT_ROOT and choose a "
            "new output directory before changing scorer settings."
        )
else:
    atomic_write_json(SCORING_PROTOCOL_PATH, scoring_protocol_payload)

TASK_OUTPUT_LOOKUP = {
    output.task_name: output
    for outputs in TASK_OUTPUTS_BY_BENCHMARK.values()
    for output in outputs
}
print("Scoring protocol SHA256:", SCORING_PROTOCOL_SHA256)


Scoring protocol SHA256: c7a183e160c05c760d91eaeb0204e3e69a966d849791683a2eb0a29094e4c075


## 15. Benchmark scoring functions

Multiple-choice metrics are reconstructed from every saved likelihood. GSM8K uses both
official frozen filters; TruthfulQA retains and normalizes the entire MC2 answer set; IFEval
runs the pinned deterministic instruction validators. HumanEval concatenates the original
function prompt and raw completion exactly as the canonical harness does.


In [15]:
def mean_and_stderr(values):
    values = [float(value) for value in values]
    if not values:
        return None, None
    mean_value = float(np.mean(values))
    stderr = float(np.std(values, ddof=1) / math.sqrt(len(values))) if len(values) > 1 else 0.0
    return mean_value, stderr


def ordered_raw_records(model_dir, benchmark):
    paths = benchmark_paths(model_dir, benchmark)
    records = repair_and_read_jsonl(paths["samples"])
    record_by_id = {record["sample_id"]: record for record in records}
    expected_order = [item["sample_id"] for item in EXPECTED_INVENTORIES[benchmark]]
    if set(record_by_id) != set(expected_order):
        raise RuntimeError(f"Cannot score incomplete {benchmark} raw artifacts.")
    return [record_by_id[sample_id] for sample_id in expected_order]


def source_doc_and_task(record):
    output = TASK_OUTPUT_LOOKUP.get(record["task_name"])
    if output is None:
        raise KeyError(f"Unknown frozen task in raw record: {record['task_name']}")
    doc_id = int(record["metadata"]["doc_id"])
    return output.task.eval_docs[doc_id], output.task


def base_score_record(record, source_sha256, metrics, details=None):
    return {
        "suite": SUITE_NAME,
        "evaluation_suite": EVALUATION_SUITE_NAME,
        "protocol_sha256": PROTOCOL_SHA256,
        "scoring_protocol_sha256": SCORING_PROTOCOL_SHA256,
        "source_samples_sha256": source_sha256,
        "model_name": record["model_name"],
        "benchmark": record["benchmark"],
        "task_name": record["task_name"],
        "sample_id": record["sample_id"],
        "subject": record.get("subject"),
        "metrics": jsonable(metrics),
        "details": jsonable(details or {}),
        "scored_at": utc_now(),
    }


def score_multiple_choice_record(record, source_sha256):
    doc, task = source_doc_and_task(record)
    choices = sorted(record["prediction"]["choices"], key=lambda item: int(item["index"]))
    responses = [(float(item["loglikelihood"]), bool(item["is_greedy"])) for item in choices]
    metrics = task.process_results(doc, responses)
    lls = np.asarray([item[0] for item in responses], dtype=np.float64)
    harness_choices = [str(item) for item in task.doc_to_choice(doc)]
    char_lengths = np.asarray([float(len(item)) for item in harness_choices], dtype=np.float64)
    predicted_index = int(np.argmax(lls))
    details = {
        "predicted_index": predicted_index,
        "predicted_label": choices[predicted_index]["label"],
        "length_normalized_predicted_index": int(np.argmax(lls / char_lengths)),
        "ground_truth_index": record["reference"].get("ground_truth_index"),
        "all_loglikelihoods_used": True,
    }
    if record["benchmark"] == "truthfulqa":
        shifted = lls - np.max(lls)
        probabilities = np.exp(shifted) / np.exp(shifted).sum()
        details["candidate_probabilities"] = [float(item) for item in probabilities]
        details["candidate_truth_labels"] = record["reference"]["candidate_truth_labels"]
    return base_score_record(record, source_sha256, metrics, details)


GSM8K_STRICT_PATTERN = re.compile(r"#### (\-?[0-9\.\,]+)")
GSM8K_FLEXIBLE_PATTERN = re.compile(r"(-?[$0-9.,]{2,})|(-?[0-9]+)")


def gsm8k_extract(raw_text, flexible=False):
    if flexible:
        matches = GSM8K_FLEXIBLE_PATTERN.findall(raw_text)
        if not matches:
            return "[invalid]"
        selected = matches[-1]
        if isinstance(selected, tuple):
            selected = next((item for item in selected if item), "[invalid]")
        return selected.strip()
    matches = GSM8K_STRICT_PATTERN.findall(raw_text)
    return matches[0].strip() if matches else "[invalid]"


def score_gsm8k_record(record, source_sha256):
    doc, task = source_doc_and_task(record)
    raw_text = record["prediction"]["raw_text"]
    strict_prediction = gsm8k_extract(raw_text, flexible=False)
    flexible_prediction = gsm8k_extract(raw_text, flexible=True)
    strict = task.process_results(doc, [strict_prediction])
    flexible = task.process_results(doc, [flexible_prediction])
    metrics = {
        "strict_exact_match": float(strict["exact_match"]),
        "flexible_exact_match": float(flexible["exact_match"]),
    }
    details = {
        "strict_extracted_answer": strict_prediction,
        "flexible_extracted_answer": flexible_prediction,
        "reference_answer": doc["answer"],
    }
    return base_score_record(record, source_sha256, metrics, details)


def score_ifeval_record(record, source_sha256):
    doc, task = source_doc_and_task(record)
    raw_metrics = task.process_results(doc, [record["prediction"]["raw_text"]])
    metrics = {
        "prompt_level_strict_accuracy": bool(raw_metrics["prompt_level_strict_acc"]),
        "instruction_level_strict_results": [bool(item) for item in raw_metrics["inst_level_strict_acc"]],
        "prompt_level_loose_accuracy": bool(raw_metrics["prompt_level_loose_acc"]),
        "instruction_level_loose_results": [bool(item) for item in raw_metrics["inst_level_loose_acc"]],
    }
    return base_score_record(
        record,
        source_sha256,
        metrics,
        {"instruction_count": len(doc["instruction_id_list"])},
    )


def aggregate_score_rows(benchmark, rows):
    if benchmark == "mmlu":
        values = [row["metrics"]["acc"] for row in rows]
        accuracy, stderr = mean_and_stderr(values)
        by_subject = {}
        for subject in sorted({row["subject"] for row in rows}):
            subject_values = [row["metrics"]["acc"] for row in rows if row["subject"] == subject]
            subject_accuracy, subject_stderr = mean_and_stderr(subject_values)
            by_subject[subject] = {
                "accuracy": subject_accuracy,
                "stderr": subject_stderr,
                "samples": len(subject_values),
            }
        return {
            "accuracy": accuracy,
            "accuracy_stderr": stderr,
            "macro_subject_accuracy": float(np.mean([item["accuracy"] for item in by_subject.values()])),
            "subjects": by_subject,
        }
    if benchmark == "gsm8k":
        output = {}
        for key in ["strict_exact_match", "flexible_exact_match"]:
            value, stderr = mean_and_stderr([row["metrics"][key] for row in rows])
            output[key] = value
            output[f"{key}_stderr"] = stderr
        return output
    if benchmark == "gpqa":
        accuracy, accuracy_stderr = mean_and_stderr([row["metrics"]["acc"] for row in rows])
        norm, norm_stderr = mean_and_stderr([row["metrics"]["acc_norm"] for row in rows])
        return {
            "accuracy": accuracy,
            "accuracy_stderr": accuracy_stderr,
            "length_normalized_accuracy": norm,
            "length_normalized_accuracy_stderr": norm_stderr,
        }
    if benchmark == "truthfulqa":
        value, stderr = mean_and_stderr([row["metrics"]["acc"] for row in rows])
        return {"mc2_accuracy": value, "mc2_accuracy_stderr": stderr}
    if benchmark == "ifeval":
        prompt_strict = [row["metrics"]["prompt_level_strict_accuracy"] for row in rows]
        prompt_loose = [row["metrics"]["prompt_level_loose_accuracy"] for row in rows]
        inst_strict = [
            value
            for row in rows
            for value in row["metrics"]["instruction_level_strict_results"]
        ]
        inst_loose = [
            value
            for row in rows
            for value in row["metrics"]["instruction_level_loose_results"]
        ]
        result = {}
        for key, values in {
            "prompt_level_strict_accuracy": prompt_strict,
            "instruction_level_strict_accuracy": inst_strict,
            "prompt_level_loose_accuracy": prompt_loose,
            "instruction_level_loose_accuracy": inst_loose,
        }.items():
            value, stderr = mean_and_stderr(values)
            result[key] = value
            result[f"{key}_stderr"] = stderr
        result["instruction_instances"] = len(inst_strict)
        return result
    raise KeyError(benchmark)


def score_nonexecuting_benchmark(model_dir, benchmark, source_sha256):
    records = ordered_raw_records(model_dir, benchmark)
    rows = []
    for record in tqdm(records, desc=f"Scoring {benchmark}", unit="sample"):
        if benchmark in {"mmlu", "gpqa", "truthfulqa"}:
            rows.append(score_multiple_choice_record(record, source_sha256))
        elif benchmark == "gsm8k":
            rows.append(score_gsm8k_record(record, source_sha256))
        elif benchmark == "ifeval":
            rows.append(score_ifeval_record(record, source_sha256))
        else:
            raise KeyError(benchmark)
    return rows, aggregate_score_rows(benchmark, rows)


## 16. HumanEval execution and crash-safe scoring runner

**Security boundary:** HumanEval necessarily executes model-generated Python. The official
evaluator applies a reliability guard and per-attempt timeout, but explicitly is not a
hardened sandbox. Before any execution, this cell creates `tidpo_generations_raw.zip`,
removes the Hugging Face token from the worker environment and ephemeral token cache,
and runs only after every model has been unloaded from CUDA memory.


In [16]:
RAW_BACKUP_PATH = OUTPUT_ROOT_PATH.parent / f"{OUTPUT_ROOT_PATH.name}_raw.zip"


def create_raw_backup():
    temporary = RAW_BACKUP_PATH.with_suffix(".zip.tmp")
    if temporary.exists():
        temporary.unlink()
    excluded_root_files = {
        "scoring_protocol.json",
        "leaderboard.json",
        "leaderboard.csv",
        "evaluation_validation.json",
    }
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED, allowZip64=True) as archive:
        for path in sorted(OUTPUT_ROOT_PATH.rglob("*")):
            if not path.is_file():
                continue
            relative = path.relative_to(OUTPUT_ROOT_PATH)
            if "scores" in relative.parts or relative.name == "evaluation_summary.json":
                continue
            if len(relative.parts) == 1 and relative.name in excluded_root_files:
                continue
            archive.write(path, arcname=str(Path(OUTPUT_ROOT_PATH.name) / relative))
    os.replace(temporary, RAW_BACKUP_PATH)
    print(f"Raw pre-execution backup: {RAW_BACKUP_PATH}")
    print(f"Raw backup SHA256: {sha256_file(RAW_BACKUP_PATH)}")


def score_humaneval(model_dir, source_sha256):
    if not EXECUTE_HUMANEVAL:
        raise RuntimeError("HumanEval execution is disabled by EXECUTE_HUMANEVAL=False.")
    records = ordered_raw_records(model_dir, "humaneval")
    predictions = []
    references = []
    for record in records:
        doc, task = source_doc_and_task(record)
        predictions.append([doc["prompt"] + record["prediction"]["raw_text"]])
        references.append(str(task.doc_to_target(doc)))

    code_eval_metric = evaluate.load(
        "code_eval",
        module_type="metric",
        revision=CODE_EVAL_REVISION,
    )

    sensitive_env_names = ["HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"]
    for name in sensitive_env_names:
        os.environ.pop(name, None)
    allow_code_eval_before = os.environ.get("HF_ALLOW_CODE_EVAL")
    token_paths = {
        Path(os.environ["HF_HOME"]) / "token",
        Path.home() / ".cache" / "huggingface" / "token",
    }
    for token_path in token_paths:
        if token_path.is_file():
            token_path.unlink()

    try:
        os.environ["HF_ALLOW_CODE_EVAL"] = "1"
        pass_at_k, granular_results = code_eval_metric.compute(
            references=references,
            predictions=predictions,
            k=[1],
            num_workers=int(HUMANEVAL_NUM_WORKERS),
            timeout=float(HUMANEVAL_TIMEOUT_SECONDS),
        )
    finally:
        if allow_code_eval_before is None:
            os.environ.pop("HF_ALLOW_CODE_EVAL", None)
        else:
            os.environ["HF_ALLOW_CODE_EVAL"] = allow_code_eval_before

    rows = []
    for task_index, record in enumerate(records):
        attempts = granular_results.get(task_index, granular_results.get(str(task_index), []))
        if len(attempts) != 1:
            raise RuntimeError(
                f"Expected one HumanEval attempt for task index {task_index}; received {len(attempts)}."
            )
        completion_id, attempt = attempts[0]
        rows.append(
            base_score_record(
                record,
                source_sha256,
                {"passed": bool(attempt["passed"])},
                {
                    "completion_id": int(completion_id),
                    "execution_result": str(attempt["result"]),
                    "timeout_seconds": float(HUMANEVAL_TIMEOUT_SECONDS),
                    "code_eval_revision": CODE_EVAL_REVISION,
                    "candidate_is_original_prompt_plus_raw_completion": True,
                    "hf_credentials_removed_before_execution": True,
                },
            )
        )
    empirical_pass_at_1, stderr = mean_and_stderr([row["metrics"]["passed"] for row in rows])
    reported_pass_at_1 = float(pass_at_k["pass@1"])
    if not math.isclose(empirical_pass_at_1, reported_pass_at_1, rel_tol=0.0, abs_tol=1e-12):
        raise RuntimeError("HumanEval per-sample results disagree with code_eval pass@1.")
    return rows, {
        "pass_at_1": reported_pass_at_1,
        "pass_at_1_stderr": stderr,
        "passed": int(sum(row["metrics"]["passed"] for row in rows)),
        "code_eval_revision": CODE_EVAL_REVISION,
        "timeout_seconds": float(HUMANEVAL_TIMEOUT_SECONDS),
        "num_workers": int(HUMANEVAL_NUM_WORKERS),
        "generated_code_executed": True,
    }


def score_paths(model_dir, benchmark):
    directory = Path(model_dir) / "scores" / benchmark
    return {
        "dir": directory,
        "per_sample": directory / "per_sample.jsonl",
        "metrics": directory / "metrics.json",
        "done": directory / "DONE",
        "incomplete": directory / "INCOMPLETE.json",
    }


def evaluation_is_complete(model_dir, benchmark, source_sha256):
    paths = score_paths(model_dir, benchmark)
    if not paths["done"].is_file() or not paths["per_sample"].is_file() or not paths["metrics"].is_file():
        return False, None
    try:
        done = json.loads(paths["done"].read_text(encoding="utf-8"))
        rows = repair_and_read_jsonl(paths["per_sample"])
        valid = (
            done.get("status") == "complete"
            and done.get("scoring_protocol_sha256") == SCORING_PROTOCOL_SHA256
            and done.get("source_samples_sha256") == source_sha256
            and done.get("samples") == len(EXPECTED_IDS[benchmark]) == len(rows)
            and set(row["sample_id"] for row in rows) == EXPECTED_IDS[benchmark]
            and done.get("per_sample_sha256") == sha256_file(paths["per_sample"])
            and done.get("metrics_sha256") == sha256_file(paths["metrics"])
        )
        return bool(valid), done
    except Exception as exc:
        warnings.warn(f"Ignoring invalid scoring DONE marker for {benchmark}: {exc}")
        return False, None


def write_score_artifacts(model_dir, benchmark, rows, metrics, source_sha256):
    paths = score_paths(model_dir, benchmark)
    paths["dir"].mkdir(parents=True, exist_ok=True)
    row_ids = [row.get("sample_id") for row in rows]
    if len(rows) != len(EXPECTED_IDS[benchmark]) or set(row_ids) != EXPECTED_IDS[benchmark]:
        raise RuntimeError(
            f"Refusing to finalize incomplete {benchmark} scores: "
            f"{len(rows)}/{len(EXPECTED_IDS[benchmark])} rows."
        )
    if len(row_ids) != len(set(row_ids)):
        raise RuntimeError(f"Refusing to finalize duplicate {benchmark} score rows.")
    metrics_payload = {
        "suite": SUITE_NAME,
        "evaluation_suite": EVALUATION_SUITE_NAME,
        "protocol_sha256": PROTOCOL_SHA256,
        "scoring_protocol_sha256": SCORING_PROTOCOL_SHA256,
        "source_samples_sha256": source_sha256,
        "benchmark": benchmark,
        "samples": len(rows),
        "metrics": metrics,
        "scored_at": utc_now(),
    }
    atomic_write_jsonl(paths["per_sample"], rows)
    atomic_write_json(paths["metrics"], metrics_payload)
    done_payload = {
        "status": "complete",
        "benchmark": benchmark,
        "samples": len(rows),
        "scoring_protocol_sha256": SCORING_PROTOCOL_SHA256,
        "source_samples_sha256": source_sha256,
        "per_sample_sha256": sha256_file(paths["per_sample"]),
        "metrics_sha256": sha256_file(paths["metrics"]),
        "completed_at": utc_now(),
    }
    atomic_write_json(paths["done"], done_payload)
    if paths["incomplete"].exists():
        paths["incomplete"].unlink()
    return done_payload


def update_evaluation_manifest(model_name, benchmark, status, samples, paths, source_sha256=None, error=None):
    MANIFEST.setdefault("evaluation_entries", [])
    MANIFEST["evaluation_entries"] = [
        entry
        for entry in MANIFEST["evaluation_entries"]
        if not (entry.get("model") == model_name and entry.get("benchmark") == benchmark)
    ]
    entry = {
        "model": model_name,
        "benchmark": benchmark,
        "status": status,
        "samples": int(samples),
        "scoring_protocol_sha256": SCORING_PROTOCOL_SHA256,
        "source_samples_sha256": source_sha256,
        "per_sample_file": str(paths["per_sample"].relative_to(OUTPUT_ROOT_PATH)),
        "metrics_file": str(paths["metrics"].relative_to(OUTPUT_ROOT_PATH)),
    }
    if paths["per_sample"].is_file():
        entry["per_sample_sha256"] = sha256_file(paths["per_sample"])
    if paths["metrics"].is_file():
        entry["metrics_sha256"] = sha256_file(paths["metrics"])
    if error:
        entry["error"] = error
    MANIFEST["evaluation_entries"].append(entry)
    MANIFEST["evaluation_entries"].sort(
        key=lambda item: (item["model"], BENCHMARK_ORDER.index(item["benchmark"]))
    )
    MANIFEST["updated_at"] = utc_now()
    atomic_write_json(MANIFEST_PATH, MANIFEST)


MANIFEST = load_manifest()
if RUN_SCORING and EXECUTE_HUMANEVAL:
    create_raw_backup()

for configured_model in MODELS:
    model_name = configured_model["name"]
    model_dir = OUTPUT_ROOT_PATH / safe_slug(model_name)
    context = {
        "model_name": model_name,
        "model_path": str(resolve_model_path(configured_model["path"])),
        "model_dir": str(model_dir),
    }
    for benchmark in BENCHMARK_ORDER:
        paths = score_paths(model_dir, benchmark)
        raw_paths = benchmark_paths(model_dir, benchmark)
        raw_complete, raw_info = benchmark_is_complete(context, benchmark)
        if not RUN_SCORING:
            update_evaluation_manifest(model_name, benchmark, "disabled", 0, paths)
            continue
        if not raw_complete:
            update_evaluation_manifest(
                model_name,
                benchmark,
                "not_ready",
                0,
                paths,
                error="Raw generation artifact is incomplete.",
            )
            continue
        source_sha256 = sha256_file(raw_paths["samples"])
        score_complete, score_info = evaluation_is_complete(model_dir, benchmark, source_sha256)
        if score_complete:
            print(f"SKIP SCORE {model_name} / {benchmark}: integrity checks passed.")
            update_evaluation_manifest(
                model_name,
                benchmark,
                "complete",
                score_info["samples"],
                paths,
                source_sha256=source_sha256,
            )
            continue
        if benchmark == "humaneval" and not EXECUTE_HUMANEVAL:
            update_evaluation_manifest(
                model_name,
                benchmark,
                "disabled",
                0,
                paths,
                source_sha256=source_sha256,
                error="EXECUTE_HUMANEVAL=False",
            )
            continue
        try:
            print(f"SCORE {model_name} / {benchmark}")
            if benchmark == "humaneval":
                rows, metrics = score_humaneval(model_dir, source_sha256)
            else:
                rows, metrics = score_nonexecuting_benchmark(model_dir, benchmark, source_sha256)
            done = write_score_artifacts(model_dir, benchmark, rows, metrics, source_sha256)
            update_evaluation_manifest(
                model_name,
                benchmark,
                "complete",
                done["samples"],
                paths,
                source_sha256=source_sha256,
            )
            print(json.dumps({"model": model_name, "benchmark": benchmark, **metrics}, indent=2))
        except Exception as exc:
            paths["dir"].mkdir(parents=True, exist_ok=True)
            error_payload = {
                "status": "incomplete",
                "model": model_name,
                "benchmark": benchmark,
                "error_type": type(exc).__name__,
                "error": str(exc),
                "source_samples_sha256": source_sha256,
                "scoring_protocol_sha256": SCORING_PROTOCOL_SHA256,
                "updated_at": utc_now(),
            }
            atomic_write_json(paths["incomplete"], error_payload)
            update_evaluation_manifest(
                model_name,
                benchmark,
                "incomplete",
                0,
                paths,
                source_sha256=source_sha256,
                error=f"{type(exc).__name__}: {exc}",
            )
            print(f"SCORING ERROR {model_name} / {benchmark}: {type(exc).__name__}: {exc}")
            traceback.print_exc()


Raw pre-execution backup: /kaggle/working/tidpo_generations_raw.zip
Raw backup SHA256: e08a12b6dc3d7a594edb634835ce9d76ab4464fbb77bf2bd27376ed56c0fb7c3


## 17. Evaluation validation and leaderboard

Every score is tied to the SHA256 of its source JSONL and scoring protocol. The leaderboard
reports each benchmark's native primary metrics separately; it does not invent a composite
score across unrelated benchmarks.


In [17]:
LEADERBOARD_METRICS = {
    "mmlu": ["accuracy"],
    "gsm8k": ["strict_exact_match", "flexible_exact_match"],
    "gpqa": ["accuracy", "length_normalized_accuracy"],
    "humaneval": ["pass_at_1"],
    "truthfulqa": ["mc2_accuracy"],
    "ifeval": [
        "prompt_level_strict_accuracy",
        "instruction_level_strict_accuracy",
        "prompt_level_loose_accuracy",
        "instruction_level_loose_accuracy",
    ],
}
EVALUATION_VALIDATION = []
LEADERBOARD = []
for configured_model in MODELS:
    model_name = configured_model["name"]
    model_dir = OUTPUT_ROOT_PATH / safe_slug(model_name)
    row = {"model": model_name}
    benchmark_summaries = {}
    for benchmark in BENCHMARK_ORDER:
        paths = score_paths(model_dir, benchmark)
        raw_paths = benchmark_paths(model_dir, benchmark)
        source_sha256 = sha256_file(raw_paths["samples"]) if raw_paths["samples"].is_file() else None
        complete, info = (
            evaluation_is_complete(model_dir, benchmark, source_sha256)
            if source_sha256
            else (False, None)
        )
        validation_row = {
            "model": model_name,
            "benchmark": benchmark,
            "status": "complete" if complete else "incomplete",
            "samples": (info or {}).get("samples", 0),
            "expected": len(EXPECTED_IDS[benchmark]),
            "source_samples_sha256": source_sha256,
        }
        EVALUATION_VALIDATION.append(validation_row)
        print(
            f"EVAL {model_name} | {benchmark}: {validation_row['status']} "
            f"({validation_row['samples']}/{validation_row['expected']})"
        )
        if not complete:
            continue
        metrics_payload = json.loads(paths["metrics"].read_text(encoding="utf-8"))
        metrics = metrics_payload["metrics"]
        benchmark_summaries[benchmark] = metrics_payload
        for metric_name in LEADERBOARD_METRICS[benchmark]:
            row[f"{benchmark}.{metric_name}"] = metrics.get(metric_name)
    atomic_write_json(
        model_dir / "evaluation_summary.json",
        {
            "suite": SUITE_NAME,
            "evaluation_suite": EVALUATION_SUITE_NAME,
            "model": model_name,
            "protocol_sha256": PROTOCOL_SHA256,
            "scoring_protocol_sha256": SCORING_PROTOCOL_SHA256,
            "benchmarks": benchmark_summaries,
            "created_at": utc_now(),
        },
    )
    LEADERBOARD.append(row)

atomic_write_json(OUTPUT_ROOT_PATH / "evaluation_validation.json", EVALUATION_VALIDATION)
atomic_write_json(
    OUTPUT_ROOT_PATH / "leaderboard.json",
    {
        "suite": SUITE_NAME,
        "evaluation_suite": EVALUATION_SUITE_NAME,
        "scoring_protocol_sha256": SCORING_PROTOCOL_SHA256,
        "rows": LEADERBOARD,
    },
)
leaderboard_columns = ["model"] + [
    f"{benchmark}.{metric}"
    for benchmark in BENCHMARK_ORDER
    for metric in LEADERBOARD_METRICS[benchmark]
]
leaderboard_csv_path = OUTPUT_ROOT_PATH / "leaderboard.csv"
leaderboard_csv_tmp = leaderboard_csv_path.with_suffix(".csv.tmp")
with leaderboard_csv_tmp.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=leaderboard_columns, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(LEADERBOARD)
os.replace(leaderboard_csv_tmp, leaderboard_csv_path)
print(json.dumps(LEADERBOARD, indent=2))


EVAL VPDPO_B_Norm_DPO | mmlu: incomplete (0/14042)
EVAL VPDPO_B_Norm_DPO | gsm8k: incomplete (0/1319)
EVAL VPDPO_B_Norm_DPO | gpqa: incomplete (0/448)
EVAL VPDPO_B_Norm_DPO | humaneval: incomplete (0/164)
EVAL VPDPO_B_Norm_DPO | truthfulqa: incomplete (0/817)
EVAL VPDPO_B_Norm_DPO | ifeval: incomplete (0/541)
EVAL VPDPO_B_Norm_VDPO | mmlu: incomplete (0/14042)
EVAL VPDPO_B_Norm_VDPO | gsm8k: incomplete (0/1319)
EVAL VPDPO_B_Norm_VDPO | gpqa: incomplete (0/448)
EVAL VPDPO_B_Norm_VDPO | humaneval: incomplete (0/164)
EVAL VPDPO_B_Norm_VDPO | truthfulqa: incomplete (0/817)
EVAL VPDPO_B_Norm_VDPO | ifeval: incomplete (0/541)
EVAL VPDPO_B_DPO | mmlu: incomplete (0/14042)
EVAL VPDPO_B_DPO | gsm8k: incomplete (0/1319)
EVAL VPDPO_B_DPO | gpqa: incomplete (0/448)
EVAL VPDPO_B_DPO | humaneval: incomplete (0/164)
EVAL VPDPO_B_DPO | truthfulqa: incomplete (0/817)
EVAL VPDPO_B_DPO | ifeval: incomplete (0/541)
EVAL VPDPO_B_VDPO | mmlu: incomplete (0/14042)
EVAL VPDPO_B_VDPO | gsm8k: incomplete (0/131

## 18. Final manifest

`manifest.json` contains separate raw-generation and evaluation entries. Each evaluation
entry records its source hash, scorer hash, status, sample count, and score-file hashes.


In [18]:
MANIFEST = load_manifest()
MANIFEST["validation"] = FINAL_VALIDATION
MANIFEST["evaluation_validation"] = EVALUATION_VALIDATION
MANIFEST["scoring_protocol_sha256"] = SCORING_PROTOCOL_SHA256
complete_evaluations = [
    entry for entry in MANIFEST.get("evaluation_entries", []) if entry.get("status") == "complete"
]
human_eval_executions = [
    entry for entry in complete_evaluations if entry.get("benchmark") == "humaneval"
]
MANIFEST["scoring_performed"] = bool(complete_evaluations)
MANIFEST["human_eval_code_executed"] = bool(human_eval_executions)
MANIFEST["raw_pre_execution_backup"] = (
    {"file": str(RAW_BACKUP_PATH), "sha256": sha256_file(RAW_BACKUP_PATH)}
    if RAW_BACKUP_PATH.is_file()
    else None
)
MANIFEST["finalized_at"] = utc_now()
atomic_write_json(MANIFEST_PATH, MANIFEST)

for configured_model in MODELS:
    metadata_path = OUTPUT_ROOT_PATH / safe_slug(configured_model["name"]) / "metadata.json"
    if not metadata_path.is_file():
        continue
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    model_evaluations = [
        entry
        for entry in MANIFEST.get("evaluation_entries", [])
        if entry.get("model") == configured_model["name"] and entry.get("status") == "complete"
    ]
    metadata["scoring_performed"] = bool(model_evaluations)
    metadata["human_eval_code_executed"] = any(
        entry.get("benchmark") == "humaneval" for entry in model_evaluations
    )
    metadata["scoring_protocol_sha256"] = SCORING_PROTOCOL_SHA256
    metadata["code_eval_revision"] = CODE_EVAL_REVISION
    metadata["evaluation_updated_at"] = utc_now()
    atomic_write_json(metadata_path, metadata)

print(json.dumps(MANIFEST, indent=2))


{
  "suite": "tidpo_compatible_v1",
  "protocol_sha256": "51f5acedf7c1742f18ed4cdbd614d14dd4fc7a4dbe09783e44f5bd2cbf409301",
  "created_at": "2026-08-29T06:43:59.764522+00:00",
  "models": [
    "VPDPO_B_Norm_DPO",
    "VPDPO_B_Norm_VDPO",
    "VPDPO_B_DPO",
    "VPDPO_B_VDPO",
    "VPDPO_C_DPO",
    "VPDPO_C_VDPO",
    "Simple_DPO",
    "VPDPO_A",
    "SimPO",
    "SAMPO",
    "TIDPO"
  ],
  "entries": [
    {
      "model": "SAMPO",
      "benchmark": "mmlu",
      "status": "incomplete",
      "samples": 0,
      "sha256": null,
      "file": "SAMPO/mmlu/samples.jsonl",
      "error": "ValueError: Tokenizer class TokenizersBackend does not exist or is not currently imported."
    },
    {
      "model": "SAMPO",
      "benchmark": "gsm8k",
      "status": "incomplete",
      "samples": 0,
      "sha256": null,
      "file": "SAMPO/gsm8k/samples.jsonl",
      "error": "ValueError: Tokenizer class TokenizersBackend does not exist or is not currently imported."
    },
    {
      "mode

## 19. ZIP export

The complete directory is archived for download. It includes frozen raw artifacts, all
scores, leaderboard files, protocols, metadata, integrity markers, and the final
manifest—never model weights or Hugging Face caches.


In [19]:
ZIP_PATH = OUTPUT_ROOT_PATH.parent / f"{OUTPUT_ROOT_PATH.name}.zip"
temporary_base = OUTPUT_ROOT_PATH.parent / f".{OUTPUT_ROOT_PATH.name}_export"
temporary_zip = Path(str(temporary_base) + ".zip")
if temporary_zip.exists():
    temporary_zip.unlink()
shutil.make_archive(
    str(temporary_base),
    "zip",
    root_dir=OUTPUT_ROOT_PATH.parent,
    base_dir=OUTPUT_ROOT_PATH.name,
)
os.replace(temporary_zip, ZIP_PATH)
print("=" * 80)
print(f"DOWNLOAD THIS FILE: {ZIP_PATH}")
print(f"ZIP SHA256: {sha256_file(ZIP_PATH)}")
if RAW_BACKUP_PATH.is_file():
    print(f"RAW PRE-EXECUTION BACKUP: {RAW_BACKUP_PATH}")
    print(f"RAW BACKUP SHA256: {sha256_file(RAW_BACKUP_PATH)}")
print("=" * 80)


DOWNLOAD THIS FILE: /kaggle/working/tidpo_generations.zip
ZIP SHA256: c54e8eceb05af5806f4d0536e146a458bd4045031abde48728c1aa4c181ec537
RAW PRE-EXECUTION BACKUP: /kaggle/working/tidpo_generations_raw.zip
RAW BACKUP SHA256: e08a12b6dc3d7a594edb634835ce9d76ab4464fbb77bf2bd27376ed56c0fb7c3
